## Credit Card Fraud Detection
This code demonstrates the end to end machine learning used to identify fraudulent credit card transactions using transactions behaviour and risk indicators


## Importing Important Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
print('kernel alive')



## Dataset Loading


In [ ]:
df = pd.read_csv("credit_card_fraud_10k.csv")
df.head()

df.info()
print("\nFraud distribution:")
print(df['is_fraud'].value_counts(normalize=True))


This data is highly imbalanced, which is often a challenge in real world fraud detection systems

## Data Preprocessing


In [ ]:
X = df.drop(['is_fraud', 'transaction_id'], axis=1)
y = df['is_fraud']
numeric_features = [
    'amount',
    'transaction_hour',
    'device_trust_score',
    'velocity_last_24h',
    'cardholder_age'
]

categorical_features = [
    'merchant_category',
    'foreign_transaction',
    'location_mismatch'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


## Model Training-Baseline Model

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("Logistic Regression Performance:")
print(classification_report(y_test, y_pred_lr))


## Model Training-Final Model

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("Random Forest Performance:")
print(classification_report(y_test, y_pred_rf))


## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix – Random Forest")
plt.show()


## Conclusion
The Random Forest model outperforms the baseline Logistic Regression model, particularly in terms of recall for fraudulent transactions. 